## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print("Libraries loaded successfully!")

## 7.1 Advanced Optimizers

Beyond vanilla gradient descent, advanced optimizers adapt learning rates per parameter.

### Momentum
$$v_t = \beta v_{t-1} + (1-\beta) \nabla J$$
$$\theta_t = \theta_{t-1} - \alpha v_t$$

### RMSprop
$$s_t = \beta s_{t-1} + (1-\beta) (\nabla J)^2$$
$$\theta_t = \theta_{t-1} - \alpha \frac{\nabla J}{\sqrt{s_t + \epsilon}}$$

### Adam (Adaptive Moment Estimation)
Combines momentum and RMSprop

In [ ]:
class AdamOptimizer:
    """
    Adam optimizer implementation.
    """
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.t = 0
        self.m = {}
        self.v = {}
    
    def update(self, params, grads):
        """
        Update parameters using Adam.
        """
        self.t += 1
        
        for key in params.keys():
            # Initialize momentum and velocity
            if key not in self.m:
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])
            
            # Update biased first moment estimate
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]
            
            # Update biased second raw moment estimate
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (grads[key]**2)
            
            # Bias correction
            m_hat = self.m[key] / (1 - self.beta1**self.t)
            v_hat = self.v[key] / (1 - self.beta2**self.t)
            
            # Update parameters
            params[key] -= self.learning_rate * m_hat / (np.sqrt(v_hat) + self.epsilon)
        
        return params

print("Adam optimizer implemented!")

## 7.2 Batch Normalization

Normalize layer inputs to stabilize training.

**Algorithm**:
1. Compute batch mean: $\mu_B = \frac{1}{m}\sum_{i=1}^m z^{(i)}$
2. Compute batch variance: $\sigma_B^2 = \frac{1}{m}\sum_{i=1}^m (z^{(i)} - \mu_B)^2$
3. Normalize: $\hat{z}^{(i)} = \frac{z^{(i)} - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$
4. Scale and shift: $\tilde{z}^{(i)} = \gamma \hat{z}^{(i)} + \beta$

In [ ]:
class BatchNormalization:
    """
    Batch Normalization layer.
    """
    def __init__(self, num_features, epsilon=1e-5, momentum=0.9):
        self.num_features = num_features
        self.epsilon = epsilon
        self.momentum = momentum
        
        # Learnable parameters
        self.gamma = np.ones((num_features, 1))
        self.beta = np.zeros((num_features, 1))
        
        # Running statistics (for inference)
        self.running_mean = np.zeros((num_features, 1))
        self.running_var = np.ones((num_features, 1))
        
        # Cache for backprop
        self.cache = {}
    
    def forward(self, Z, training=True):
        """
        Forward pass.
        Z: (num_features, m)
        """
        if training:
            # Batch statistics
            mu = np.mean(Z, axis=1, keepdims=True)
            var = np.var(Z, axis=1, keepdims=True)
            
            # Update running statistics
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * mu
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * var
        else:
            # Use running statistics during inference
            mu = self.running_mean
            var = self.running_var
        
        # Normalize
        Z_norm = (Z - mu) / np.sqrt(var + self.epsilon)
        
        # Scale and shift
        out = self.gamma * Z_norm + self.beta
        
        # Cache for backprop
        self.cache = {'Z': Z, 'Z_norm': Z_norm, 'mu': mu, 'var': var}
        
        return out
    
    def backward(self, dout):
        """
        Backward pass.
        """
        Z = self.cache['Z']
        Z_norm = self.cache['Z_norm']
        mu = self.cache['mu']
        var = self.cache['var']
        m = Z.shape[1]
        
        # Gradients of learnable parameters
        dgamma = np.sum(dout * Z_norm, axis=1, keepdims=True)
        dbeta = np.sum(dout, axis=1, keepdims=True)
        
        # Gradient of normalized Z
        dZ_norm = dout * self.gamma
        
        # Gradient of Z (chain rule through normalization)
        std = np.sqrt(var + self.epsilon)
        dZ = (1 / m) * (1 / std) * (
            m * dZ_norm - np.sum(dZ_norm, axis=1, keepdims=True) - 
            Z_norm * np.sum(dZ_norm * Z_norm, axis=1, keepdims=True)
        )
        
        return dZ, dgamma, dbeta

print("Batch Normalization implemented!")

## 7.3 Dropout Regularization

Randomly "drop" neurons during training to prevent overfitting.

**Training**: Set neuron to 0 with probability $p$
**Inference**: Scale activations by $(1-p)$

In [ ]:
class Dropout:
    """
    Dropout regularization layer.
    """
    def __init__(self, keep_prob=0.8):
        self.keep_prob = keep_prob
        self.mask = None
    
    def forward(self, A, training=True):
        """
        Apply dropout during training.
        """
        if training:
            # Create dropout mask
            self.mask = (np.random.rand(*A.shape) < self.keep_prob).astype(float)
            # Apply mask and scale
            return A * self.mask / self.keep_prob
        else:
            # No dropout during inference
            return A
    
    def backward(self, dA):
        """
        Backprop through dropout.
        """
        return dA * self.mask / self.keep_prob

print("Dropout implemented!")

## 7.4 Advanced Neural Network with All Features

Let's build a complete neural network with:
- Batch normalization
- Dropout
- Adam optimizer
- Learning rate decay

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

class AdvancedNeuralNetwork:
    """
    Advanced neural network with modern techniques.
    """
    def __init__(self, layers_dims, learning_rate=0.001, 
                 use_batchnorm=True, dropout_prob=0.8):
        """
        layers_dims: list of layer sizes [input, hidden1, hidden2, ..., output]
        """
        self.layers_dims = layers_dims
        self.L = len(layers_dims) - 1
        self.use_batchnorm = use_batchnorm
        
        # Initialize parameters
        self.params = {}
        for l in range(1, self.L + 1):
            # He initialization
            self.params[f'W{l}'] = np.random.randn(layers_dims[l], layers_dims[l-1]) * \
                                    np.sqrt(2.0 / layers_dims[l-1])
            self.params[f'b{l}'] = np.zeros((layers_dims[l], 1))
        
        # Batch normalization layers
        self.bn_layers = {}
        if use_batchnorm:
            for l in range(1, self.L):
                self.bn_layers[f'bn{l}'] = BatchNormalization(layers_dims[l])
        
        # Dropout layers
        self.dropout_layers = {}
        for l in range(1, self.L):
            self.dropout_layers[f'dropout{l}'] = Dropout(dropout_prob)
        
        # Adam optimizer
        self.optimizer = AdamOptimizer(learning_rate)
        
        self.cache = {}
        self.costs = []
    
    def forward(self, X, training=True):
        """
        Forward propagation with batch norm and dropout.
        """
        A = X
        self.cache['A0'] = X
        
        # Hidden layers
        for l in range(1, self.L):
            Z = self.params[f'W{l}'] @ A + self.params[f'b{l}']
            self.cache[f'Z{l}'] = Z
            
            # Batch normalization
            if self.use_batchnorm:
                Z = self.bn_layers[f'bn{l}'].forward(Z, training)
            
            # Activation
            A = relu(Z)
            self.cache[f'A{l}_pre_dropout'] = A
            
            # Dropout
            A = self.dropout_layers[f'dropout{l}'].forward(A, training)
            self.cache[f'A{l}'] = A
        
        # Output layer (no batchnorm or dropout)
        Z = self.params[f'W{self.L}'] @ A + self.params[f'b{self.L}']
        self.cache[f'Z{self.L}'] = Z
        A = sigmoid(Z)
        self.cache[f'A{self.L}'] = A
        
        return A
    
    def compute_cost(self, Y):
        """
        Binary cross-entropy cost.
        """
        m = Y.shape[1]
        A = self.cache[f'A{self.L}']
        A_clipped = np.clip(A, 1e-10, 1 - 1e-10)
        cost = -1/m * np.sum(Y * np.log(A_clipped) + (1 - Y) * np.log(1 - A_clipped))
        return cost
    
    def backward(self, Y):
        """
        Backpropagation.
        """
        m = Y.shape[1]
        grads = {}
        
        # Output layer
        dA = self.cache[f'A{self.L}'] - Y
        dZ = dA
        grads[f'W{self.L}'] = 1/m * dZ @ self.cache[f'A{self.L-1}'].T
        grads[f'b{self.L}'] = 1/m * np.sum(dZ, axis=1, keepdims=True)
        dA = self.params[f'W{self.L}'].T @ dZ
        
        # Hidden layers (backward)
        for l in reversed(range(1, self.L)):
            # Dropout backward
            dA = self.dropout_layers[f'dropout{l}'].backward(dA)
            
            # ReLU backward
            dZ = dA * relu_derivative(self.cache[f'Z{l}'])
            
            # Batch norm backward
            if self.use_batchnorm:
                dZ, dgamma, dbeta = self.bn_layers[f'bn{l}'].backward(dZ)
                grads[f'gamma{l}'] = dgamma
                grads[f'beta{l}'] = dbeta
            
            # Linear backward
            grads[f'W{l}'] = 1/m * dZ @ self.cache[f'A{l-1}'].T
            grads[f'b{l}'] = 1/m * np.sum(dZ, axis=1, keepdims=True)
            
            if l > 1:
                dA = self.params[f'W{l}'].T @ dZ
        
        return grads
    
    def train(self, X, Y, num_epochs=100, print_cost=False):
        """
        Train the network.
        """
        for epoch in range(num_epochs):
            # Forward
            A = self.forward(X, training=True)
            
            # Cost
            cost = self.compute_cost(Y)
            self.costs.append(cost)
            
            # Backward
            grads = self.backward(Y)
            
            # Update with Adam
            self.params = self.optimizer.update(self.params, grads)
            
            if print_cost and epoch % 100 == 0:
                print(f"Epoch {epoch}: Cost = {cost:.4f}")
    
    def predict(self, X):
        """
        Predict (inference mode).
        """
        A = self.forward(X, training=False)
        return (A > 0.5).astype(int)

print("Advanced Neural Network implemented!")

## 7.5 Training on Digits Dataset

In [ ]:
# Load and prepare data
digits = load_digits()
X_all = digits.data
y_all = digits.target

# Binary classification: 0 vs 1
mask = (y_all == 0) | (y_all == 1)
X_binary = X_all[mask]
y_binary = y_all[mask]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X_binary, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Transpose for neural network
X_train_T = X_train_scaled.T
X_test_T = X_test_scaled.T
Y_train = y_train.reshape(1, -1)
Y_test = y_test.reshape(1, -1)

print(f"Training: {X_train_T.shape}, Test: {X_test_T.shape}")

# Compare: Without vs With advanced techniques
print("\n" + "="*60)
print("Training WITHOUT advanced techniques...")
print("="*60)
nn_basic = AdvancedNeuralNetwork([64, 32, 16, 1], 
                                 learning_rate=0.001,
                                 use_batchnorm=False,
                                 dropout_prob=1.0)  # No dropout
nn_basic.train(X_train_T, Y_train, num_epochs=1000, print_cost=True)

print("\n" + "="*60)
print("Training WITH advanced techniques...")
print("="*60)
nn_advanced = AdvancedNeuralNetwork([64, 32, 16, 1],
                                    learning_rate=0.001,
                                    use_batchnorm=True,
                                    dropout_prob=0.8)
nn_advanced.train(X_train_T, Y_train, num_epochs=1000, print_cost=True)

# Evaluate
train_pred_basic = nn_basic.predict(X_train_T)
test_pred_basic = nn_basic.predict(X_test_T)
train_pred_adv = nn_advanced.predict(X_train_T)
test_pred_adv = nn_advanced.predict(X_test_T)

print("\n" + "="*60)
print("RESULTS COMPARISON")
print("="*60)
print("\nBasic Network:")
print(f"  Train Accuracy: {np.mean(train_pred_basic == Y_train):.2%}")
print(f"  Test Accuracy:  {np.mean(test_pred_basic == Y_test):.2%}")
print("\nAdvanced Network (BatchNorm + Dropout + Adam):")
print(f"  Train Accuracy: {np.mean(train_pred_adv == Y_train):.2%}")
print(f"  Test Accuracy:  {np.mean(test_pred_adv == Y_test):.2%}")

# Plot cost curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(nn_basic.costs, linewidth=2, label='Basic')
axes[0].plot(nn_advanced.costs, linewidth=2, label='Advanced')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Cost', fontsize=12)
axes[0].set_title('Training Cost Comparison', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Zoom in on later epochs
axes[1].plot(nn_basic.costs[100:], linewidth=2, label='Basic')
axes[1].plot(nn_advanced.costs[100:], linewidth=2, label='Advanced')
axes[1].set_xlabel('Epoch (after 100)', fontsize=12)
axes[1].set_ylabel('Cost', fontsize=12)
axes[1].set_title('Training Cost (Zoomed)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7.6 Learning Rate Schedules

Adjust learning rate during training:

1. **Step Decay**: $\alpha_t = \alpha_0 \cdot \text{drop}^{\lfloor \frac{\text{epoch}}{\text{epochs_drop}} \rfloor}$
2. **Exponential Decay**: $\alpha_t = \alpha_0 e^{-kt}$
3. **1/t Decay**: $\alpha_t = \frac{\alpha_0}{1 + kt}$

In [ ]:
# Learning rate schedule functions
def step_decay(initial_lr, epoch, drop_every=200, drop_factor=0.5):
    return initial_lr * (drop_factor ** (epoch // drop_every))

def exponential_decay(initial_lr, epoch, k=0.005):
    return initial_lr * np.exp(-k * epoch)

def inverse_time_decay(initial_lr, epoch, k=0.5):
    return initial_lr / (1 + k * epoch / 100)

# Visualize schedules
epochs = np.arange(1000)
initial_lr = 0.01

lrs_step = [step_decay(initial_lr, e) for e in epochs]
lrs_exp = [exponential_decay(initial_lr, e) for e in epochs]
lrs_inv = [inverse_time_decay(initial_lr, e) for e in epochs]

plt.figure(figsize=(12, 7))
plt.plot(epochs, lrs_step, linewidth=2, label='Step Decay')
plt.plot(epochs, lrs_exp, linewidth=2, label='Exponential Decay')
plt.plot(epochs, lrs_inv, linewidth=2, label='Inverse Time Decay')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.title('Learning Rate Schedules', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nLearning rate schedules:")
print("- Step decay: Sudden drops at intervals")
print("- Exponential: Smooth exponential decrease")
print("- Inverse time: Fast initial decrease, then slow")

## 7.7 Optimizer Comparison

Let's compare SGD, Momentum, RMSprop, and Adam on the same problem.

In [ ]:
from sklearn.datasets import make_circles

# Generate challenging dataset
X_circles, y_circles = make_circles(n_samples=300, noise=0.15, factor=0.3, random_state=42)
X_circles_T = X_circles.T
Y_circles = y_circles.reshape(1, -1)

# Train with different optimizers
optimizers = {
    'Adam (β1=0.9, β2=0.999)': AdamOptimizer(learning_rate=0.01, beta1=0.9, beta2=0.999),
    'Adam (β1=0.8, β2=0.99)': AdamOptimizer(learning_rate=0.01, beta1=0.8, beta2=0.99),
}

results = {}

for name, optimizer in optimizers.items():
    print(f"\nTraining with {name}...")
    nn = AdvancedNeuralNetwork([2, 16, 8, 1], learning_rate=0.01,
                              use_batchnorm=False, dropout_prob=1.0)
    nn.optimizer = optimizer  # Replace optimizer
    nn.train(X_circles_T, Y_circles, num_epochs=500)
    results[name] = nn.costs

# Plot comparison
plt.figure(figsize=(12, 7))
for name, costs in results.items():
    plt.plot(costs, linewidth=2, label=name)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Cost', fontsize=12)
plt.title('Optimizer Comparison on Non-Linear Dataset', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print("\nFinal Costs:")
for name, costs in results.items():
    print(f"  {name}: {costs[-1]:.6f}")

## Key Takeaways

### 1. **Advanced Optimizers**
- **Adam** is the go-to optimizer (adaptive per-parameter learning rates)
- Combines momentum (first moment) and RMSprop (second moment)
- Default: $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$
- Faster convergence than vanilla SGD

### 2. **Batch Normalization**
- Normalizes layer inputs → faster training, higher learning rates
- Reduces internal covariate shift
- Acts as regularization (slight noise from batch statistics)
- Use **before** activation function (common practice)

### 3. **Dropout**
- Randomly drop neurons during training (p = 0.2-0.5 typical)
- Prevents co-adaptation of features
- Ensemble effect (averaging many sub-networks)
- **Critical**: Turn off during inference
- Inverted dropout: Scale during training, not inference

### 4. **Learning Rate Schedules**
- Start high (fast initial learning), decay over time (fine-tuning)
- **Step decay**: Simple, works well in practice
- **Exponential**: Smooth decrease
- **Cosine annealing**: Cyclical for better convergence

### 5. **Weight Initialization**
- **He initialization** for ReLU: $W \sim \mathcal{N}(0, \frac{2}{n_{in}})$
- **Xavier** for sigmoid/tanh: $W \sim \mathcal{N}(0, \frac{1}{n_{in}})$
- Critical for deep networks

### 6. **Modern Best Practices**
```python
# Recommended architecture:
Input
  → [Linear → BatchNorm → ReLU → Dropout] × N
  → Linear → Sigmoid/Softmax
  → Output

# Training configuration:
- Optimizer: Adam (lr=0.001)
- BatchNorm: Yes (momentum=0.9)
- Dropout: 0.2-0.5 for fully connected layers
- Initialization: He for ReLU
- Learning rate: Decay by 0.1 every N epochs
```

### 7. **When to Use What**
- **Batch Normalization**: Almost always (faster training)
- **Dropout**: When overfitting (large gap train/test accuracy)
- **Adam**: Default optimizer choice
- **Learning rate decay**: For long training (>1000 epochs)

### 8. **Common Pitfalls**
- Forgetting to turn off dropout during inference
- Using batch norm with very small batches (unstable statistics)
- Too aggressive dropout (underfitting)
- Learning rate too high (divergence) or too low (slow convergence)

### 9. **Debugging Checklist**
- ✓ Features scaled (StandardScaler)
- ✓ Proper initialization (He/Xavier)
- ✓ Learning rate reasonable (0.001-0.01 for Adam)
- ✓ Cost decreasing (if not, reduce LR)
- ✓ No NaN/Inf in gradients
- ✓ Batch norm before activation
- ✓ Dropout off during inference

## Practice Exercises

1. **Batch Normalization**: Implement from scratch. Verify forward and backward pass with gradient checking.

2. **Optimizer Comparison**: Compare SGD, Momentum, RMSprop, and Adam on MNIST. Plot convergence curves.

3. **Dropout Study**: Train networks with dropout = [0.0, 0.2, 0.5, 0.8]. How does it affect overfitting?

4. **Learning Rate Finder**: Implement LR range test. Train with exponentially increasing LR, plot loss. Find optimal LR.

5. **Deep Network**: Build 5-10 layer network. Does batch norm help with vanishing gradients?

6. **Transfer Learning**: Pre-train on large dataset, fine-tune on small dataset. Compare with training from scratch.

7. **Learning Rate Schedules**: Implement cosine annealing. Compare with step decay.

8. **Gradient Clipping**: Implement gradient clipping (max norm). Does it stabilize training?

9. **Multi-class**: Extend to 10-class MNIST. Implement softmax output and cross-entropy loss.

10. **Hyperparameter Tuning**: Use random search to find best [learning rate, hidden size, dropout, batch norm]. Report best config.

## References

1. **"Batch Normalization: Accelerating Deep Network Training"**: Ioffe & Szegedy (2015)
2. **"Dropout: A Simple Way to Prevent Neural Networks from Overfitting"**: Srivastava et al. (2014)
3. **"Adam: A Method for Stochastic Optimization"**: Kingma & Ba (2014)
4. **"Delving Deep into Rectifiers"**: He et al. (2015)
5. **"Deep Learning"**: Goodfellow et al., Chapters 7-8
6. **"Understanding the Difficulty of Training Deep Feedforward Neural Networks"**: Glorot & Bengio (2010)

---

**Next**: [Lecture 8: Clustering](08_clustering.ipynb)